# Baseline Model: Logistic Regression

Predict whether the **home team wins** using only information known before kickoff.

- **Train:** 2004-2024 seasons. 2002-2003 are left out because they have no box-score stats (they're only used for last-season records in `data_transform.ipynb`).
- **Test:** the 2025 season, which the model never sees during training.
- **Compare against two simple baselines:** always pick the home team, and pick the better-ranked team.

**Run `data_exploration.ipynb` and then `data_transform.ipynb` first.**

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay)

## Load the Pre-Game Features

In [ ]:
games = pd.read_csv('datasets/cfb_pregame_features.csv', parse_dates=['date'])

# Home-minus-away pre-game stats (built in data_transform.ipynb)
DIFF_FEATURES = [
    'diff_points_scored',
    'diff_points_allowed',
    'diff_yards_gained',
    'diff_yards_allowed',
    'diff_turnovers_lost',
    'diff_turnovers_forced',
    'diff_third_down_pct',
    'diff_penalty_yards',
    'diff_win_pct',
    'diff_last_season_win_pct',
]

# Other things known before kickoff
GAME_FEATURES = [
    'neutral',           # 1 if played at a neutral site
    'conf_game',         # 1 if both teams are in the same conference
    'rank_home_filled',  # AP rank going into the game, unranked = 30
    'rank_away_filled',
]

FEATURES = DIFF_FEATURES + GAME_FEATURES
TARGET = 'home_win'

## Split by Season

In [ ]:
train = games[(games['season'] >= 2004) & (games['season'] <= 2024)]
test = games[games['season'] == 2025]

X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print('Training games (2004-2024):', len(train))
print('Test games (2025):', len(test))
print('Home team won', round(y_train.mean() * 100, 1), '% of training games and',
      round(y_test.mean() * 100, 1), '% of test games')

## Simple Baselines

A model is only useful if it beats these:
1. **Always pick the home team.**
2. **Pick the better-ranked team.** Lower rank number is better, and unranked teams count as 30. If both teams have the same rank (usually both unranked), pick the home team.

In [ ]:
# Baseline 1: predict 1 (home team wins) for every game
always_home_pred = pd.Series(1, index=test.index)

# Baseline 2: predict 1 if the home team is ranked the same or better than the away team
better_ranked_pred = (test['rank_home_filled'] <= test['rank_away_filled']).astype(int)

## Train the Logistic Regression

The model is a pipeline of three steps:
1. `SimpleImputer` fills the few blank values (e.g. a team's first game in its first FBS season) with the median from the training data.
2. `StandardScaler` puts every feature on the same scale, so the model treats yards and percentages fairly and we can compare the coefficients later.
3. `LogisticRegression` learns how much each feature pushes the odds of a home win up or down.

In [ ]:
model = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
model.fit(X_train, y_train)

test_pred = model.predict(X_test)              # 1 = home team wins, 0 = away team wins
test_prob = model.predict_proba(X_test)[:, 1]  # model's chance that the home team wins

print('Training accuracy:', round(accuracy_score(y_train, model.predict(X_train)), 3))
print('Test accuracy:    ', round(accuracy_score(y_test, test_pred), 3))

## Results on the 2025 Season

The positive class is **home team wins**:
- **Precision:** when we pick the home team, how often it actually wins.
- **Recall:** of the games the home team actually won, how many we picked.
- **ROC-AUC:** how well the predicted probabilities rank home wins above home losses. The simple baselines only give a yes/no pick, not a probability, so they don't get one.

In [ ]:
def score_predictions(name, y_true, y_pred, y_prob=None):
    """Return one row of metrics for a set of predictions."""
    row = {
        'model': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'roc_auc': float('nan'),
    }
    if y_prob is not None:
        row['roc_auc'] = roc_auc_score(y_true, y_prob)
    return row


results = pd.DataFrame([
    score_predictions('Always pick home team', y_test, always_home_pred),
    score_predictions('Pick better-ranked team', y_test, better_ranked_pred),
    score_predictions('Logistic regression', y_test, test_pred, test_prob),
])
results.round(3)

"Always pick the home team" gets a perfect recall of 1.0 just by never picking the away team. That's why we look at accuracy, F1 and ROC-AUC together instead of any one number.

### Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, test_pred,
                                        display_labels=['Away win', 'Home win'],
                                        cmap='Blues')
plt.title('Logistic regression on 2025 games')
plt.show()

### ROC Curve

In [ ]:
RocCurveDisplay.from_predictions(y_test, test_prob, name='Logistic regression')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guessing')
plt.title('ROC curve on 2025 games')
plt.legend()
plt.show()

## Which Features Matter Most?

Because the features were scaled, the coefficients can be compared. A positive bar pushes the prediction toward the home team, a negative bar toward the away team. Features that move together (like points scored and yards gained) end up sharing credit, so read this as a rough guide.

In [ ]:
logreg = model.named_steps['logisticregression']
coefficients = pd.Series(logreg.coef_[0], index=FEATURES).sort_values()

coefficients.plot.barh(figsize=(8, 6))
plt.axvline(0, color='gray')
plt.title('Logistic regression coefficients (scaled features)')
plt.xlabel('Coefficient')
plt.show()

## Where Does the Model Miss?

Accuracy by week of the 2025 season. Week 16 is the postseason (bowls and playoffs). Week 15 is conference championship weekend, which only has a handful of games, so its bar swings a lot. The table under the chart shows how many games are behind each bar.

In [ ]:
test_results = test[['week', 'postseason']].copy()
test_results['correct'] = (test_pred == y_test).astype(int)

by_week = test_results.groupby('week')['correct'].agg(accuracy='mean', games='count')

by_week['accuracy'].plot.bar(figsize=(10, 4), label='Accuracy')
plt.axhline(test_results['correct'].mean(), color='gray', linestyle='--', label='Overall')
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('Logistic regression accuracy by week (2025)')
plt.legend()
plt.show()

by_week.T.round(2)